# End-to-end simulation of observation and pipeline reduction of Barnard's star as a prevision RV standard

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [github](https://github.com/oirlab/HISPEC_DRP).</span>


This notebook shows two examples of simulating data:

1. A simulation of a single up-the-ramp exposure of a star with a theoretical spectrum with similar properties to Bernard's star, assuming no planets.
2. A segment of a precision radial velocity observations of the same star, now with an orbiting super Earth causing the radial velocity to vary.

For each exposure, the notebook simulates a BSPEC rate map from a Phoenix template (approximately similar to Barnard's star), builds a `RampModel`, then runs `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline`.

---

## Setup

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [1]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/Users/ckilpatrick/barnards_star_prv_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


hispecdrp 0.0.post2+waveinject.local @ /Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/__init__.py
output: /Users/ckilpatrick/barnards_star_prv_example
KOA_CALIBRATION_CACHE: /Users/ckilpatrick/barnards_star_prv_example/cals/KOA_CALIBRATION_CACHE


## 1. Sync / verify KOA caldb (BSPEC)


In [2]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


[2026-08-20T15:14:35][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/barnards_star_prv_example/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/barnards_star_prv_example/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/barnards_star_prv_example/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'
[2026-08-20T15:14:35][koa_middleware.keck_client:81][INFO]: Session validation: no observer ID found; re-login required.
[2026-08-20T15:14:35][koa_middleware.keck_client:51][INFO]: No valid login detected, logging in...
[2026-08-20T15:14:37][koa_middleware.keck_client:136][INFO]: User info request successful: observer ID=4187
[2026-08-20T15:14:37][koa_middleware.keck_client:56][INFO]: Keck Observer login successful! Observer ID = 4187
[2026-08-20T15:14:37][koa_middleware.database.remote_database:35][INFO]: RemoteCalibrationDB initialized: instrument='hispec', url='https://www3.keck.hawaii.edu/api/calibra

## 2. Single exposure of star with no planet

Stellar properties are approximately similar to Bernard's star (same Teff, logg, and metallicity).

### 2.1 Simulated rate map from a 1D spectrum → `RampModel`

Load a Phoenix template, densify it, Doppler-shift the densified 1D spectrum by
`radial_velocity_km_s` (`inject_rv_via='spectrum'`, default), paint with ZEMAX, then
build a raw UTR `RampModel`. Stage-0 KOA masters plus sim Stage-1 products
(packaged line-ZP WAVE — **unscaled**, same bake for every RV — traces, fiberflat)
are registered into the local caldb.

1. Fiber 1 is the on the star assuming ideal coupling. 
2. Fiber 2 is empty (will later be updated to include etalon spectrum). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind

In [3]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,3]
READ_TIME = 1.7 # seconds

# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -110.1

exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename="lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
    mag_ab=6.15,
    filter_band="J",  # BSPEC (yJ)
    n_reads=n_reads,
    seed=0,
    radial_velocity_km_s=STELLAR_RV,
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    sky_fiber=3,
    apply_e2e_throughput=FIBERS
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")


[2026-08-20T15:14:39][hispecdrp.simulations.simulate_up_the_ramp:1620][INFO]: inject_rv_via='spectrum': densify→Doppler-paint radial_velocity_km_s=-110.100000; WAVE left at packaged line-ZP bake
[2026-08-20T15:14:39][hispecdrp.simulations.simulate_up_the_ramp:1309][INFO]: Building default rate map (fibers=[1, 3], spectrograph=BSPEC, output_dir=/Users/ckilpatrick/barnards_star_prv_example).
[2026-08-20T15:14:39][hispecdrp.primitives.utils.spectral_utils:493][INFO]: Normalized spectrum to J=6.15 (AB mag); scale=6.2912e+05 (was 20.65 mag)
[2026-08-20T15:14:39][hispecdrp.simulations.simulate_up_the_ramp:1368][INFO]: Densifying fiber=1 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-20T15:14:40][hispecdrp.simulations.simulate_up_the_ramp:1368][INFO]: Densifying fiber=3 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-20T15:14:40][hispecdrp.simulations.simulate_up_the_ramp:1385][INFO]: Applying densified-template Doppler shift (topocentric) radial_velocity_km

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


### 2.2 Stage 0 → 1 → 2 via pipeline classes


In [4]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

l0 = Stage0Pipeline.call(ramp, output_dir=OUT)

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_primitives = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline.call(l0, output_dir=OUT, primitives=stage1_primitives)

injected_m_s = float(STELLAR_RV) * 1000.0
l2 = Stage2Pipeline.call(
l1,
    output_dir=OUT,
    primitives={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {
            "ccf_input_velocity": injected_m_s,
            "do_fibers": [1],
            "save_preview": True,
            "preview_write_png": False,
        },
        "sky_sub": {"run": True},
    },
)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")

/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: invalid value encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: divide by zero encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:401: RuntimeWarning: invalid value encountered in multiply
  std_corr = np.abs(flux_corr) * np.sqrt(


[2026-08-20T15:18:25][hispecdrp.primitives.utils.ccf.ccf_utils:6404][INFO]: CCF line_mask: fiber=1 building preview n_preview_orders=19
[2026-08-20T15:18:25][hispecdrp.primitives.utils.ccf.ccf_line_mask_spline:500][INFO]: CCF preview (spline): finalize n_slots=20 n_workers=10
[2026-08-20T15:18:26][hispecdrp.primitives.utils.ccf.ccf_utils:6431][INFO]: CCF line_mask: fiber=1 finished n_chunks=19 preview=True
[2026-08-20T15:18:26][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:513][INFO]: Stellar RV: combining chunk RVs fiber=1 n_valid=19 outlier_sigma=3.00 prior=-110100.00 m/s prior_half_width=7500 m/s
[2026-08-20T15:18:26][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:549][INFO]: Stellar RV: aggregate RV fiber=1 rv_mean=-110226.71 m/s rv_std=0.96 m/s n_inlier=19 n_sane=19/19
[2026-08-20T15:18:26][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:997][INFO]: Stellar RV: RV algorithm returned fiber=1 algo=ccf_line_mask n_chunks=19
[2026-08-20T15:18:26][hispecdrp.primitives

## 3. Segment of PRV observations

We now simulate multiple exposures over several nights for a super Earth planet orbiting the same Barnard's star analog.

### 3.1 Simulating rate maps from an initial 1D spectrum and varying RV

1. Fiber 1 is the on the star assuming ideal coupling. 
2. Fiber 2 is empty (will later be updated to include etalon spectrum). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- stellar variability of any kind

In [5]:
import numpy as np


FIBERS = [1,3]
READ_TIME = 1.7 # seconds

# Injected stellar RV (km/s).
STELLAR_RV = -110.1

# observation parameters
starting_jd = 2461293.91667 # sep 10 2026
nights = 3
exposures_per_night = 3
exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

nightly_start_jd_arr = starting_jd + np.arange(0,nights)
time_offsets = np.linspace(0, (exposure_time+READ_TIME)/86400 * (exposures_per_night - 1), exposures_per_night)
obs_jd_arr = np.add.outer(nightly_start_jd_arr, time_offsets).flatten()

# planet parameters
phase0_jd = starting_jd 
semiamplitude_K = 15 # m/s
period = 1.75 # days

phase_arr = (obs_jd_arr - phase0_jd) / period
rv_arr = semiamplitude_K * np.sin(2 * np.pi * phase_arr)/1000 + STELLAR_RV


Define function to simulate and get L2 data product from each exposure

In [6]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
import gc

def run_one_injection(label: str, injected_rv_km_s: float, mjd_start: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return L2 data product."""
    run_dir = OUT
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    ramp, cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename="lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
        mag_ab=6.15,
        filter_band="J",  # BSPEC (yJ)
        n_reads=n_reads,
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via='spectrum',
        sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
        sky_fiber=3,
        apply_e2e_throughput=FIBERS,
        mjd_start=mjd_start,
        register_stage1_sims=False,
        register_wavesol=False,
        save_ramp=False,
    )
    wmeta = getattr(cals.get("wavesol"), "meta", None)
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(wmeta, 'note', None)}  "
        f"injected_rv_via={getattr(wmeta, 'injected_rv_via', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline.call(ramp, output_dir=run_dir, save_result=False)

    del ramp
    del rate_map

    l1 = Stage1Pipeline.call(
        l0,
        output_dir=run_dir,
        save_result=False,
        primitives={
            "spec_extract": {"do_fibers": do_fibers},
            "set_wavesol": {"wavesol": cals["wavesol"]},
        },
    )

    del l0

    l2 = Stage2Pipeline.call(
        l1,
        output_dir=run_dir,
        save_result=True,
        primitives={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "ccf_input_velocity": injected_m_s,
                "do_fibers": list(FIBERS),
                "save_preview": False,
                "preview_write_png": False,
            },
            "sky_sub": {"run": False},
        },
    )

    del l1
    gc.collect()

    return {
        "label": label,
        "mjd": mjd_start,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "wavesol": cals["wavesol"],
        "l2": l2,
    }


Now run

In [7]:
l2_results = {}

for i in range(nights*exposures_per_night):
    l2_results[str(obs_jd_arr[i])] = run_one_injection(label=str(obs_jd_arr[i]),
                                                       injected_rv_km_s=rv_arr[i],
                                                       mjd_start=obs_jd_arr[i],
                                                       seed=0)


=== 2461293.91667: inject -110.100000 km/s, seed=0 ===
[2026-08-20T15:18:27][hispecdrp.simulations.simulate_up_the_ramp:1620][INFO]: inject_rv_via='spectrum': densify→Doppler-paint radial_velocity_km_s=-110.100000; WAVE left at packaged line-ZP bake
[2026-08-20T15:18:27][hispecdrp.simulations.simulate_up_the_ramp:1309][INFO]: Building default rate map (fibers=[1, 3], spectrograph=BSPEC, output_dir=/Users/ckilpatrick/barnards_star_prv_example).
[2026-08-20T15:18:28][hispecdrp.primitives.utils.spectral_utils:493][INFO]: Normalized spectrum to J=6.15 (AB mag); scale=6.2912e+05 (was 20.65 mag)
[2026-08-20T15:18:28][hispecdrp.simulations.simulate_up_the_ramp:1368][INFO]: Densifying fiber=1 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-20T15:18:28][hispecdrp.simulations.simulate_up_the_ramp:1368][INFO]: Densifying fiber=3 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-20T15:18:28][hispecdrp.simulations.simulate_up_the_ramp:1385][INFO]: Applying densified

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 1 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


[2026-08-20T15:19:34][hispecdrp.calibrations.hispec_store:279][INFO]: Selecting calibration for input=None using DetectorFlatSelector(origin=ANY, cal_version=LATEST, master_cal=None)
[2026-08-20T15:19:34][hispecdrp.calibrations.hispec_store:292][INFO]: Selected calibration filename=HB.20260623.00000.00.mastercal-detflat.001.fits ID=eb660282-2902-472b-9627-55ea39053b9b
[2026-08-20T15:19:34][koa_middleware.store:432][INFO]: File found in cache for filename=HB.20260623.00000.00.mastercal-detflat.001.fits ID=eb660282-2902-472b-9627-55ea39053b9b.
[2026-08-20T15:19:35][hispecdrp.calibrations.hispec_store:279][INFO]: Selecting calibration for input=None using BiasSelector(origin=ANY, cal_version=LATEST, master_cal=None)
[2026-08-20T15:19:35][hispecdrp.calibrations.hispec_store:292][INFO]: Selected calibration filename=HB.20260623.00000.00.mastercal-bias.001.fits ID=1c581758-36a7-4a25-bbd1-a3efea8c2796
[2026-08-20T15:19:35][koa_middleware.store:432][INFO]: File found in cache for filename=HB.2

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: invalid value encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: divide by zero encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:401: RuntimeWarning: invalid value encountered in multiply
  std_corr = np.abs(flux_corr) * np.sqrt(


[2026-08-20T15:22:13][hispecdrp.primitives.utils.ccf.ccf_utils:6431][INFO]: CCF line_mask: fiber=1 finished n_chunks=19 preview=False
[2026-08-20T15:22:13][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:513][INFO]: Stellar RV: combining chunk RVs fiber=1 n_valid=19 outlier_sigma=3.00 prior=-110100.00 m/s prior_half_width=7500 m/s
[2026-08-20T15:22:13][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:549][INFO]: Stellar RV: aggregate RV fiber=1 rv_mean=-107100.14 m/s rv_std=0.88 m/s n_inlier=19 n_sane=19/19
[2026-08-20T15:22:13][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:997][INFO]: Stellar RV: RV algorithm returned fiber=1 algo=ccf_line_mask n_chunks=19
[2026-08-20T15:22:13][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1003][INFO]: Stellar RV: writing RV tables fiber=1
[2026-08-20T15:22:13][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1428][INFO]: Stellar RV: _write_rv_tables fiber=1 spectrograph=BSPEC n_per_chunk_rows=19 rv_agg=-107100.14 m/s
[2026-

... (36 outputs left out)

[2026-08-20T15:49:40][hispecdrp.calibrations.hispec_store:279][INFO]: Selecting calibration for input=None using DetectorFlatSelector(origin=ANY, cal_version=LATEST, master_cal=None)
[2026-08-20T15:49:40][hispecdrp.calibrations.hispec_store:292][INFO]: Selected calibration filename=HB.20260623.00000.00.mastercal-detflat.001.fits ID=eb660282-2902-472b-9627-55ea39053b9b
[2026-08-20T15:49:40][koa_middleware.store:432][INFO]: File found in cache for filename=HB.20260623.00000.00.mastercal-detflat.001.fits ID=eb660282-2902-472b-9627-55ea39053b9b.
[2026-08-20T15:49:40][hispecdrp.calibrations.hispec_store:279][INFO]: Selecting calibration for input=None using BiasSelector(origin=ANY, cal_version=LATEST, master_cal=None)
[2026-08-20T15:49:40][hispecdrp.calibrations.hispec_store:292][INFO]: Selected calibration filename=HB.20260623.00000.00.mastercal-bias.001.fits ID=1c581758-36a7-4a25-bbd1-a3efea8c2796
[2026-08-20T15:49:40][koa_middleware.store:432][INFO]: File found in cache for filename=HB.2

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: invalid value encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:398: RuntimeWarning: divide by zero encountered in divide
  frac_science = science_std / science_flux
/Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/primitives/blaze_correction/blaze_correction_primitive.py:401: RuntimeWarning: invalid value encountered in multiply
  std_corr = np.abs(flux_corr) * np.sqrt(


[2026-08-20T15:52:21][hispecdrp.primitives.utils.ccf.ccf_utils:6431][INFO]: CCF line_mask: fiber=1 finished n_chunks=19 preview=False
[2026-08-20T15:52:21][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:513][INFO]: Stellar RV: combining chunk RVs fiber=1 n_valid=19 outlier_sigma=3.00 prior=-110088.22 m/s prior_half_width=7500 m/s
[2026-08-20T15:52:21][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:549][INFO]: Stellar RV: aggregate RV fiber=1 rv_mean=-107088.26 m/s rv_std=0.89 m/s n_inlier=19 n_sane=19/19
[2026-08-20T15:52:21][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:997][INFO]: Stellar RV: RV algorithm returned fiber=1 algo=ccf_line_mask n_chunks=19
[2026-08-20T15:52:21][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1003][INFO]: Stellar RV: writing RV tables fiber=1
[2026-08-20T15:52:21][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1428][INFO]: Stellar RV: _write_rv_tables fiber=1 spectrograph=BSPEC n_per_chunk_rows=19 rv_agg=-107088.26 m/s
[2026-

### 3.4 Injected vs recovered RV vs MJD

Three-panel PRV diagnostic via `plot_injected_vs_recovered_rv`:

1. Raw injected and recovered RVs.
2. Residual `(rec - inj) - mean(rec - inj)` on a fixed ±1 m/s scale.
3. RV signal: injected about the systemic RV, and recovered minus a zero-point
   (mean recovered).


In [ ]:
from hispecdrp.simulations.plotting import plot_injected_vs_recovered_rv

summary = plot_injected_vs_recovered_rv(
    l2_results,
    systemic_rv_km_s=STELLAR_RV,
    fiber=1,
)
